In [1]:
import os
import re
import sys
import smtplib
import importlib
import numpy as np
from email.message import EmailMessage
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ==============================================================================
# ⚙️ CONFIGURACIÓN DE ENTORNO FORZADA Y MODELOS
# ==============================================================================
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Forzamos la carga del archivo .env apuntando directamente a su ubicación
ruta_env = ROOT_DIR / ".env"
load_dotenv(dotenv_path=ruta_env)

token = os.getenv("GITHUB_TOKEN")
base_url = os.getenv("OPENAI_BASE_URL")

# Modelo principal para el Agente
chat_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=token,
    base_url=base_url,
    temperature=0.1 # Temperatura ultra baja para que no improvise y cierre la venta sí o sí
)

# Modelo auxiliar para resúmenes
summary_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=token,
    base_url=base_url,
    temperature=0
)

# Embeddings para tu base de conocimiento fija (RAG)
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=token,
    base_url=base_url
)

# Importación dinámica de tus módulos de base de datos nativos
import Scripts.catalog_db as catalog_db
importlib.reload(catalog_db)
from Scripts.catalog_db import (
    init_catalog_db,
    query_product_db,
    query_products_by_region,
    query_shipping_to_region,
    simulate_order,
    get_all_skus,
    list_branches,
    create_order_by_branch_code,
    get_order_details,
)

# Iniciar catálogo técnico
init_catalog_db()

# ==============================================================================
# 📖 BASE DE CONOCIMIENTO (REQUISITOS MÍNIMOS ESTÁTICOS)
# ==============================================================================
documento = """
Notebook ideal para programación y gaming: Mínimo 16GB RAM, SSD 512GB, GPU RTX 3050+.
En Chile (PC Factory): Gama media: 600.000 - 900.000 CLP. Gama alta: 900.000 - 1.500.000 CLP.
"""
chunks = [documento[i:i+200] for i in range(0, len(documento), 200)]
vector_db = [{"texto": ch, "embedding": embeddings_model.embed_query(ch)} for ch in chunks]

def similitud(a, b): return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
def obtener_contexto_estatico(pregunta, top_k=1):
    emb_q = embeddings_model.embed_query(pregunta)
    resultados = sorted([(similitud(emb_q, item["embedding"]), item["texto"]) for item in vector_db], reverse=True)
    return resultados[0][1]

# ==============================================================================
# 🛠️ AUXILIARES INTERNOS DE FORMATO Y CONFIGURACIÓN SMTP
# ==============================================================================
def _format_currency(value: int) -> str:
    return f"${value:,} CLP"

def _format_block(lines: list) -> str:
    return "\n".join(lines)

def _find_first_keyword(text: str, candidates: list) -> str | None:
    normalized = text.upper()
    for candidate in candidates:
        if candidate.upper() in normalized:
            return candidate
    return None

def _get_smtp_config() -> tuple[str, str, str, int]:
    remitente = os.getenv("GMAIL_REMITENTE")
    password = os.getenv("GMAIL_APP_PASSWORD")
    server = os.getenv("SMTP_SERVER", "smtp.gmail.com")
    port = int(os.getenv("SMTP_PORT", "587"))
    return remitente, password, server, port

def _send_order_email(recipient: str, subject: str, body: str) -> None:
    remitente, password, server, port = _get_smtp_config()
    message = EmailMessage()
    message["From"] = remitente
    message["To"] = recipient
    message["Subject"] = subject
    message.set_content(body)

    with smtplib.SMTP(server, port) as smtp:
        smtp.starttls()
        smtp.login(remitente, password)
        smtp.send_message(message)

def _format_order_details(resultado: dict) -> str:
    lines = [
        f"Simulación de compra para sucursal {resultado['branch']['codigo']} ({resultado['branch']['branch_nombre']}):",
        f"Cliente: {resultado['cliente_nombre'] or 'No especificado'}",
        "Productos:",
    ]
    for item in resultado['items']:
        lines.append(
            f"- {item['cantidad']} x {item['sku']} ({item['nombre']}) @ {_format_currency(item['precio_unitario'])} c/u"
        )
    lines.append(f"Total estimado: {_format_currency(resultado['total'])}")
    return "\n".join(lines)

def _format_invoice_details(order_info: dict) -> str:
    order = order_info['order']
    lines = [
        f"Factura de Venta Real - Orden #{order['id']}",
        f"Fecha: {order['fecha']}",
        f"Sucursal de Retiro: {order['branch_codigo']} ({order['branch_nombre']})",
        "",
        "Detalle de Ítems Comprados:",
    ]
    for item in order_info['items']:
        subtotal = int(item['precio_unitario'] * item['cantidad'] * (1 - item['descuento']))
        lines.append(f"- {item['cantidad']} x {item['sku']} ({item['producto']}) | Subtotal: {_format_currency(subtotal)}")
    lines.append(f"\nTotal Final Pagado: {_format_currency(order['total'])}")
    return "\n".join(lines)

# ==============================================================================
# 🛠️ AGENTE: DECLARACIÓN DE HERRAMIENTAS CON FILTRADO NATIVO ESTRICTO
# ==============================================================================

@tool
def revisar_requisitos_teoricos_notebooks(consulta: str) -> str:
    """Usa esta herramienta cuando el usuario pida recomendaciones generales sobre notebooks."""
    return obtener_contexto_estatico(consulta)

@tool
def consultar_producto_avanzado(termino_busqueda: str) -> str:
    """
    Busca un producto en la base de datos usando su SKU o palabras clave de su nombre.
    Si el producto está agotado o no existe, interroga la lista nativa de SKUs para extraer alternativas reales
    de la misma categoría exacta (Procesadores o GPUs) y obligar al bot a ofrecer opciones correctas en stock.
    """
    query = termino_busqueda.strip()
    if not query: 
        return "Por favor, proporciona un término de búsqueda válido."

    sku, producto = query_product_db(query)
    
    # --- 🔄 FILTRADO NATIVO: SI NO EXISTE O TIENE STOCK 0, EXTRAEMOS DE TU CATALOG_DB ---
    if not producto or producto.get("total_stock", 0) == 0:
        lines = []
        if not producto:
            lines.append(f"El producto exacto '{termino_busqueda}' no figura en el catálogo actual de la base de datos.")
        else:
            lines.append(f"El producto '{producto['nombre']}' ({sku}) se encuentra agotado temporalmente.")
            
        lines.append("\n🔍 ALTERNATIVAS EXCLUSIVAS DE LA MISMA CATEGORÍA DISPONIBLES CON STOCK:")
        
        # Obtenemos de forma nativa todos tus SKUs cargados en Scripts/catalog_db.py
        todos_los_skus = get_all_skus()
        
        # Identificamos el tipo de componente consultado
        es_procesador = any(w in query.lower() for w in ["ryzen", "intel", "cpu", "procesador", "7800x3d", "14700k"])
        es_gpu = any(w in query.lower() for w in ["rtx", "gpu", "video", "nvidia", "amd", "4070"])
        
        alternativas_encontradas = []
        
        for s in todos_los_skus:
            # Traemos la info de cada SKU usando tu función nativa
            _, prod_info = query_product_db(s)
            if prod_info and prod_info.get("total_stock", 0) > 0:
                if es_procesador and "CPU" in s:
                    alternativas_encontradas.append((s, prod_info))
                elif es_gpu and "GPU" in s:
                    alternativas_encontradas.append((s, prod_info))
                    
            if len(alternativas_encontradas) >= 3:
                break
                
        # Si la consulta no encajó en CPU/GPU o no había filtros, armamos un mix balanceado del catálogo real
        if not alternativas_encontradas:
            for s in todos_los_skus[:4]:
                _, prod_info = query_product_db(s)
                if prod_info and prod_info.get("total_stock", 0) > 0:
                    alternativas_encontradas.append((s, prod_info))

        # Renderizamos la lista numerada estricta para que el agente la exponga
        for idx, (alt_sku, info) in enumerate(alternativas_encontradas, start=1):
            precio_lista = info["precio_lista"]
            lines.append(f"{idx}. **{alt_sku}** - {info['nombre']} | Precio: {_format_currency(precio_lista)}")
            
        lines.append("\nIMPORTANTE: Presenta esta lista enumerada exacta. El cliente seleccionará una opción de esta lista para proceder a la facturación real.")
        return _format_block(lines)

    # --- FLUJO NORMAL SI EL PRODUCTO CONSULTADO SÍ TIENE EXISTENCIAS ---
    precio_lista = producto["precio_lista"]
    desc_porcentaje = producto["descuento_efectivo"]
    precio_efectivo = precio_lista - int(precio_lista * desc_porcentaje)

    lines = [
        f"DATOS DEL CATÁLOGO ({sku}):",
        f"- Componente: {producto['nombre']}",
        f"- Precio Lista: {_format_currency(precio_lista)}",
        f"- Precio Efectivo: {_format_currency(precio_efectivo)} ({int(desc_porcentaje * 100)}% de Descuento)",
        f"- Stock Consolidado: {producto['total_stock']} unidades.",
        "- Disponibilidad en Sucursales:"
    ]
    
    inventario = producto.get("inventory", [])
    if inventario:
        for item in inventory:
            lines.append(f"  - [{item['branch_codigo']}] {item['branch_nombre']}: {item['cantidad']} unidades.")
    return _format_block(lines)

@tool
def confirmar_compra_por_correo(orden_texto: str) -> str:
    """
    Procesa la compra real de un producto en base a un texto libre que incluya sucursal y SKU.
    Descuenta stock de la base de datos y despacha la factura al correo de prueba del .env de forma automatizada.
    """
    texto = orden_texto.strip()
    branch_candidates = [branch["codigo"] for branch in list_branches()]
    sku_candidates = get_all_skus()
    
    branch_code = _find_first_keyword(texto, branch_candidates)
    sku = _find_first_keyword(texto, sku_candidates)
    cantidad_match = re.search(r"(\d+)", texto)
    email_match = re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", texto)
    
    cantidad = int(cantidad_match.group(1)) if cantidad_match else 1
    recipient = email_match.group(0) if email_match else os.getenv("GMAIL_REMITENTE")

    if not branch_code or not sku:
        return "No pude procesar la compra. Asegúrate de indicar la sucursal (ej: SCL-CENTRO) y el SKU de la alternativa seleccionada."

    try:
        # Llama a tu función nativa de registro e impacto en inventarios
        order_id = create_order_by_branch_code(branch_code, [(sku, cantidad)], cliente_nombre="Diego Soto")
        order_info = get_order_details(order_id)
    except ValueError as e:
        return f"Error en la base de datos: {e}."

    invoice_body = _format_invoice_details(order_info)
    subject = f"Factura de Venta PC Factory - Orden #{order_id}"
    
    try:
        _send_order_email(recipient, subject, invoice_body)
        status_email = f"📧 Factura electrónica despachada con éxito vía SMTP a tu casilla: {recipient}"
    except Exception as e:
        status_email = f"⚠️ Pedido ingresado en tablas SQL, pero falló la conexión SMTP de Gmail: {e}"
    
    return _format_block([
        f"✅ ¡VENTA EJECUTADA Y CONFIRMADA EN EL SISTEMA!",
        f"- Factura Emitida: Orden #{order_id}",
        f"- Punto de Retiro Validado: {branch_code}",
        f"- Total Cobrado: {_format_currency(order_info['order']['total'])}",
        status_email
    ])

tools = [revisar_requisitos_teoricos_notebooks, consultar_producto_avanzado, confirmar_compra_por_correo]

# ==============================================================================
# 🧠 PROMPT COMERCIAL ORIENTADO A RESOLUCIÓN TRANSACCIONAL OBLIGATORIA
# ==============================================================================
prompt_sistema = """Eres el asesor virtual y cerrador de ventas experto de la tienda PC Factory Chile.
Tu objetivo principal es asistir al usuario cruzando sus intenciones con tus herramientas y asegurar la emisión de la factura si el cliente manifiesta intenciones de compra.

REGLAS ABSOLUTAS DE TRABAJO CON HISTORIAL AUTOMÁTICO:
1. Si el cliente pregunta por un producto que no está disponible (ej: Ryzen 7800X3D), la herramienta 'consultar_producto_avanzado' te entregará una lista de alternativas reales del mismo tipo (ej: CPU-AMD-R5-5600X). Expón esa lista numerada de forma clara y exacta en tu mensaje.
2. Si el cliente en el siguiente mensaje automatizado te dice "me interesa la segunda opción de procesador" o selecciona un ítem por su orden, asocia de forma inteligente a qué SKU de tu mensaje anterior corresponde esa opción.
3. Invoca inmediatamente la herramienta 'confirmar_compra_por_correo' pasando el SKU correcto mapeado de la lista de alternativas y la sucursal que el cliente indicó (ej: SCL-CENTRO). No pidas confirmaciones adicionales ni detengas la conversación; procesa la venta y emite la factura de forma obligatoria.
4. Si el cliente no provee un email, la herramienta tomará por defecto el correo de pruebas de la configuración interna, así que procede de todas formas.
5. Formatea los valores en pesos chilenos ($169,990 CLP).
"""

app_agente = create_react_agent(chat_llm, tools, prompt=prompt_sistema)

# ==============================================================================
# 🧠 MEMORIA DE HISTORIAL
# ==============================================================================
store = {}
def get_session_history(session_id: str):
    if session_id not in store: 
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# ==============================================================================
# 🚀 EJECUCIÓN DEL CIRCUITO TRANSACCIONAL COMPLETO
# ==============================================================================
def ejecutar_chatbot_agente():
    session_id = "hilo_simulacion_factura_obligatoria"
    history = get_session_history(session_id)

    # El flujo de simulación automatizado exacto que necesitas validar
    inputs = [
        "Hola, ¿tienen stock del procesador Ryzen 7800X3D?",
        "Perfecto, me interesa la segunda opción de procesador que me diste. Quiero confirmar la compra de 1 unidad de esa alternativa para retirar en SCL-CENTRO y que me envíen la factura."
    ]

    for user_input in inputs:
        print(f"\n➡️ Usuario: {user_input}")
        history.add_user_message(user_input)
        
        try:
            response = app_agente.invoke(
                {"messages": history.messages}, 
                config={"configurable": {"thread_id": session_id}}
            )
            respuesta_final = response["messages"][-1].content
            print(f"🤖 Agente PC Factory:\n{respuesta_final}")
            history.add_ai_message(respuesta_final)
        except Exception as e:
            print(f"❌ Error operativo: {e}")
        print("-" * 50)

if __name__ == "__main__":
    ejecutar_chatbot_agente()

C:\Users\diego\AppData\Local\Temp\ipykernel_30216\286732356.py:302: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app_agente = create_react_agent(chat_llm, tools, prompt=prompt_sistema)



➡️ Usuario: Hola, ¿tienen stock del procesador Ryzen 7800X3D?
🤖 Agente PC Factory:
El procesador Ryzen 7800X3D no está disponible, pero aquí tienes algunas alternativas que tenemos en stock:

1. **CPU-AMD-R5-5600X** - Procesador AMD Ryzen 5 5600X (6 Núcleos / 12 Hilos) | Precio: $169,990 CLP
2. **CPU-AMD-R7-5700X** - Procesador AMD Ryzen 7 5700X (8 Núcleos / 16 Hilos) | Precio: $219,990 CLP
3. **CPU-AMD-R7-7800X3D** - Procesador AMD Ryzen 7 7800X3D (El rey del Gaming) | Precio: $449,990 CLP

Por favor, indícame cuál de estas opciones te interesa.
--------------------------------------------------

➡️ Usuario: Perfecto, me interesa la segunda opción de procesador que me diste. Quiero confirmar la compra de 1 unidad de esa alternativa para retirar en SCL-CENTRO y que me envíen la factura.
🤖 Agente PC Factory:
¡Tu compra ha sido confirmada exitosamente! Aquí tienes los detalles:

- **Producto:** 1 unidad de CPU-AMD-R7-5700X
- **Punto de Retiro:** SCL-CENTRO
- **Total Cobrado:** $197,991 